# M3C4 — Level 3: Feature Selection Exercises

**Purpose:** implement and compare three feature sets.

**Do:** save a copy, run from top to bottom, and upload `telco_source.csv` and `M3C4_Runtime.zip` together when asked. Local Jupyter: keep both files beside this notebook. Complete each TODO; ask for help if needed.

**Check:** choose `top_k` before validation. The final test remains closed. After a reset, start again at setup; rerun all following cells after any change.

## Setup

**Do:** run the provided preparation. It fits preprocessing on train only.

**Check:** 4,225 train rows, 1,409 validation rows, 1,409 reserved test rows and 50 prepared columns.

In [ ]:
#@title ▶ Start the lesson
from pathlib import Path
from zipfile import ZipFile, BadZipFile
comparison = results = rfe_columns = None  # Invalidate old results before a new start.
required = ['telco_source.csv', 'M3C4_Runtime.zip']
if any(not Path(name).exists() for name in required):
    try:
        from google.colab import files
    except ImportError:
        raise FileNotFoundError('Place telco_source.csv and M3C4_Runtime.zip beside this notebook, then rerun C01.') from None
    print('Select BOTH telco_source.csv and M3C4_Runtime.zip from the lesson package.')
    files.upload()
missing = [name for name in required if not Path(name).exists()]
if missing:
    raise FileNotFoundError('Missing: ' + ', '.join(missing) + '. Rerun C01 with the issued files and original names.')
try:
    with ZipFile('M3C4_Runtime.zip') as support:
        folder = Path('_m3c4_runtime'); folder.mkdir(exist_ok=True)
        for name in ['m3c4_runtime.py', 'M3_Split_Card.json']:
            (folder / name).write_bytes(support.read(name))
except (BadZipFile, KeyError):
    raise ValueError('Wrong or damaged M3C4_Runtime.zip. Upload the issued ZIP again, then rerun C01.') from None
exec(compile((folder / 'm3c4_runtime.py').read_text(encoding='utf-8'), 'm3c4_runtime.py', 'exec'), globals())

## 1 — Correlation filter

**Purpose:** detect overlapping numeric inputs before one-hot encoding.

**Do:** implement `correlation_filter`. Use the nine fields in `numeric`, in their declared order, from `train_features` only. Find pairs with strictly `abs(r) > 0.85`. Process pairs in order: keep the earlier field, remove the later one; if either was already removed, record the pair as already covered. Exclude removed numeric names (`num__` prefix) from the prepared candidates. Preserve the full 50-column baseline.

**Check:** return `(pairs, decisions, candidates)`: DataFrames with columns `A, B, r` and `A, B, Removed, Reason`, and an ordered list of candidate names. Expect 47 candidates. Inspect the actual pairs and decisions.

In [ ]:
def correlation_filter(train_numeric, prepared_names):
    # TODO: return pairs, decisions, candidates.
    raise NotImplementedError("Implement correlation_filter before running the next cell.")

In [ ]:
pairs, decisions, candidates = correlation_filter(train_features[numeric], list(X_train.columns))
assert isinstance(pairs, pd.DataFrame) and list(pairs.columns) == ['A', 'B', 'r']
assert isinstance(decisions, pd.DataFrame) and list(decisions.columns) == ['A', 'B', 'Removed', 'Reason']
assert len(X_train.columns) == 50 and len(candidates) == 47 and len(set(candidates)) == 47
assert set(candidates).issubset(X_train.columns)
show_table('Correlation pairs · train numeric', 'Strictly abs(r) > 0.85.', pairs)
show_table('Correlation decisions', 'Earlier fields take priority; covered pairs need no second removal.', decisions)
show_note('Candidate count', f'{len(candidates)} candidates; all {len(X_train.columns)} baseline columns remain available.')

## 2 — Mutual Information

**Purpose:** rank the filtered candidates using train only.

**Do:** implement `rank_mi`. Pass an explicit discrete mask: one-hot columns, `num__SeniorCitizen`, and `num__num_services` are discrete. Factorize those columns into integer categories before `mutual_info_classif`; use seed 42. Return a descending, stable `pd.Series` indexed by feature name.

**Check:** every candidate appears once and scores are finite. Choose your own integer `top_k` and explain it **before** running validation.

In [ ]:
def rank_mi(X, y, candidate_names):
    # TODO: return a descending Series of train MI scores.
    raise NotImplementedError("Implement rank_mi before running the next cell.")

In [ ]:
mi_scores = rank_mi(X_train, train_y, candidates)
assert isinstance(mi_scores, pd.Series) and len(mi_scores) == len(candidates)
assert mi_scores.index.is_unique and set(mi_scores.index) == set(candidates)
assert np.isfinite(mi_scores.to_numpy()).all() and (mi_scores >= 0).all() and mi_scores.is_monotonic_decreasing
show_table('Train MI ranking', 'Higher individual MI does not guarantee a better combination.', mi_scores.rename_axis('Feature').reset_index(name='Train MI'), max_height=420)

## 3 — Choose before validation

**Do:** set `top_k` to your own integer between 1 and the number of candidates; write your reason. Select the first `top_k` names.

**Check:** your selection has exactly `top_k` distinct names. Do not repeatedly tune on validation.

In [ ]:
top_k = None  # TODO: your integer choice before validation
selection_reason = ''  # TODO: your own reason
assert isinstance(top_k, int) and not isinstance(top_k, bool) and 1 <= top_k <= len(candidates), 'Choose a valid integer top_k.'
assert isinstance(selection_reason, str) and selection_reason.strip(), 'Write your reason.'
mi_columns = list(mi_scores.index[:top_k])
show_table('MI selection', 'Selected before validation.', pd.DataFrame({'Feature': mi_columns}), max_height=360)

## 4 — RFE

**Do:** implement `select_rfe` on train candidates using `RFE` with Random Forest 100 trees, seed 42, `step=1`, and `max(1, floor(candidate count / 2))` kept features.

**Check:** return `(selected, ranks)`: an ordered list of selected names and a DataFrame with `Feature, Selected, Rank`. Expect 23 selected from 47. This can take time.

In [ ]:
def select_rfe(X, y, candidate_names):
    # TODO: fit RFE and return selected names and ranks.
    raise NotImplementedError("Implement select_rfe before running the next cell.")

In [ ]:
start = time.perf_counter()
rfe_columns, rfe_ranks = select_rfe(X_train, train_y, candidates)
rfe_seconds = time.perf_counter() - start
assert len(rfe_columns) == max(1, len(candidates)//2) and len(set(rfe_columns)) == len(rfe_columns)
assert set(rfe_columns).issubset(candidates)
assert isinstance(rfe_ranks, pd.DataFrame) and list(rfe_ranks.columns) == ['Feature', 'Selected', 'Rank'] and len(rfe_ranks) == len(candidates)
assert list(rfe_ranks['Feature']) == candidates and rfe_ranks['Feature'].is_unique
assert pd.api.types.is_bool_dtype(rfe_ranks['Selected'])
assert pd.api.types.is_integer_dtype(rfe_ranks['Rank']) and (rfe_ranks['Rank'] > 0).all()
assert rfe_ranks['Rank'].eq(1).equals(rfe_ranks['Selected'])
assert set(rfe_ranks.loc[rfe_ranks['Selected'], 'Feature']) == set(rfe_columns)
show_table('RFE ranks', 'Selected=True and rank 1 identify the kept features.', rfe_ranks, max_height=420)
show_note('RFE summary', f'Kept {len(rfe_columns)} of {len(candidates)}; runtime {rfe_seconds:.2f} seconds.\nMI ∩ RFE: ' + ', '.join(sorted(set(mi_columns) & set(rfe_columns))))

## 5 — Compare on validation

**Do:** implement `compare_sets`. Fit a **new** Random Forest (100 trees, seed 42) for each of all 50, MI and RFE. Use the same training rows and validation rows for all three; calculate Accuracy and F1 with `Yes` as the positive class.

**Check:** return a DataFrame with columns `Feature Set, Num Features, Accuracy, F1`, in order `all, MI, RFE`. Each score must be between 0 and 1. The final test stays closed.

In [ ]:
def compare_sets(X_train, X_validation, y_train, y_validation, mi_columns, rfe_columns):
    # TODO: return one result row per feature set.
    raise NotImplementedError("Implement compare_sets before running the next cell.")

In [ ]:
results = compare_sets(X_train, X_validation, train_y, validation_y, mi_columns, rfe_columns)
assert isinstance(results, pd.DataFrame) and list(results.columns) == ['Feature Set', 'Num Features', 'Accuracy', 'F1']
assert list(results['Feature Set']) == ['all', 'MI', 'RFE']
assert list(results['Num Features']) == [50, top_k, len(rfe_columns)]
assert results[['Accuracy', 'F1']].notna().all().all() and results[['Accuracy', 'F1']].ge(0).all().all() and results[['Accuracy', 'F1']].le(1).all().all()
show_table('Validation comparison', 'Same 1,409 validation customers for each set.', results)
show_note('Accuracy gap', f"Largest gap: {round((results.Accuracy.max()-results.Accuracy.min()) * len(validation_y))} customer predictions.")

## 6 — Explain and save

**Do:** write your answers in your own words using the actual outputs.

**One correlation pair, the removal decision and its limitation:** …

**Your recommendation, feature counts, Accuracy and F1:** …

**MI/RFE overlap and one disagreement, if any:** …

**What does the validation accuracy gap mean in customer predictions?** …

**If MonthlyCharges is selected, what can the results say about churn, and why do they not prove that charges cause churn?** …

**For new data, what must be learned again on training data (or inside each cross-validation fold), and when should top_k be chosen?** …

**Check:** save the current notebook in Drive or download the `.ipynb`; reopen it and check your answers and results. The checks above validate output shape and basic constraints, not the correctness of your implementation or understanding.